In [1]:
# q1 embedding a query
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)
v1[0]

2026-06-29 03:12:57.192255917 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


np.float64(-0.02058203437252893)

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [3]:
# q2 cosine similarity
q1 = "How does approximate nearest neighbor search work?"
doc_map = {document["filename"]: document["content"] for document in documents}

target_doc = doc_map["02-vector-search/lessons/07-sqlitesearch-vector.md"]
target_v = embed.encode(target_doc)

v1 = embed.encode(q1)
v1.dot(target_v)


np.float64(0.36107027225589694)

In [4]:
# q3 chunking

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
c_vectors = [{'filename': chunk['filename'], 'vector': embed.encode(chunk['content'])} for chunk in chunks]
scores = [{'filename': c_vector['filename'], 'score': c_vector['vector'].dot(v1)} for c_vector in c_vectors]
scores.sort(key=lambda x: x['score'], reverse=True)
scores[0]

{'filename': '02-vector-search/lessons/07-sqlitesearch-vector.md',
 'score': np.float64(0.6489017718578812)}

In [7]:
# q4 vector search with minsearch

from tqdm.auto import tqdm
import numpy as np

# batch_size = 1
# vectors = []

# texts = []
# print(documents[0])

# for doc in documents:
#     text = doc['content']
#     texts.append(text)

# print(len(texts))

# for i in tqdm(range(0, len(texts), batch_size)):
#     batch = texts[i:i + batch_size]
#     batch_vectors = embed.encode(batch)
#     vectors.extend(batch_vectors)

# print(len(vectors))

X = np.array([c_vector["vector"] for c_vector in c_vectors])

from minsearch import VectorSearch
vindex = VectorSearch()
vindex.fit(X, chunks)


In [9]:
q2 = "What metric do we use to evaluate a search engine?"
v2 = embed.encode(q2)
print(v2)

results = vindex.search(v2, num_results=5)

[-3.45251120e-02 -4.76347907e-02 -9.52288143e-02  2.40308090e-02
 -1.80039094e-02  3.22336786e-02  4.11009181e-04  2.86892654e-02
  2.71596053e-02 -6.36475643e-02 -3.86707619e-02 -5.52097101e-02
  3.84640827e-02  3.63455416e-02 -9.32710316e-02 -5.29945155e-02
  8.20461574e-02 -5.03414745e-03  2.50910438e-02 -8.70656696e-02
  7.20100754e-02  1.29639035e-02  1.16451658e-01 -8.20982256e-02
 -6.52023347e-03 -4.00227584e-02 -8.36716391e-02 -1.24123525e-02
 -1.90502106e-02 -6.24077402e-03 -3.73876830e-02  6.12152591e-03
  5.60926979e-02  6.56479763e-02 -6.34850137e-02  3.43527274e-02
 -3.35941763e-02 -3.51476658e-02  2.54281298e-02 -9.75776093e-03
 -4.84886223e-02 -2.35025387e-02 -2.90664668e-02  3.65187041e-02
  2.23322831e-02 -3.29535923e-02 -4.82611746e-02  8.59540277e-03
 -3.70969933e-03  5.49367302e-02 -9.72623872e-02 -1.93188061e-04
 -1.25472844e-02 -2.15638746e-02 -5.50835867e-03  4.47906698e-02
 -5.21629946e-02 -2.24940001e-02 -2.30708396e-02 -8.72699683e-02
  9.89726366e-03 -5.08535

TypeError: float() argument must be a string or a real number, not 'dict'

In [ ]:
v2.dot(dv)


np.float64(0.01973045838992233)

In [ ]:
from ingest import load_faq_data

documents = load_faq_data()

In [ ]:
# texts = [doc["question"] + " " + doc["answer"] for doc in documents]
texts = [doc["content"] for doc in documents]

KeyError: 'content'

In [ ]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}